### Scenario 2: A cross-functional team with one data scientist working on an ML model

MLflow setup:

* tracking server: yes, local server
* backend store: sqlite database
* artifacts store: local filesystem

The experiments can be explored locally by accessing the local tracking server.

In [4]:
import mlflow
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score

mlflow.set_tracking_uri("http://127.0.0.1:5000")

In [2]:
print(f"tracking URI: '{mlflow.get_tracking_uri()}'")

tracking URI: 'http://127.0.0.1:5000'


In [3]:
mlflow.search_experiments()

[<Experiment: artifact_location='mlflow-artifacts:/0', creation_time=1720181144202, experiment_id='0', last_update_time=1720181144202, lifecycle_stage='active', name='Default', tags={}>]

In [5]:
mlflow.set_experiment("my-experiment-1")

with mlflow.start_run():

    X, y = load_iris(return_X_y=True)

    params = {"C": 0.1, "random_state": 42}
    mlflow.log_params(params)

    lr = LogisticRegression(**params).fit(X, y)
    y_pred = lr.predict(X)
    mlflow.log_metric("accuracy", accuracy_score(y, y_pred))

    mlflow.sklearn.log_model(lr, artifact_path="models")
    print(f"default artifacts URI: '{mlflow.get_artifact_uri()}'")

2024/07/05 14:07:36 INFO mlflow.tracking.fluent: Experiment with name 'my-experiment-1' does not exist. Creating a new experiment.


default artifacts URI: 'mlflow-artifacts:/1/bbfb1cdb47ff41b6942e24c27958df24/artifacts'


In [6]:
mlflow.search_experiments()

[<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1720181256921, experiment_id='1', last_update_time=1720181256921, lifecycle_stage='active', name='my-experiment-1', tags={}>,
 <Experiment: artifact_location='mlflow-artifacts:/0', creation_time=1720181144202, experiment_id='0', last_update_time=1720181144202, lifecycle_stage='active', name='Default', tags={}>]

### Interacting with the model registry

In [7]:
from mlflow.tracking import MlflowClient


client = MlflowClient("http://127.0.0.1:5000")

In [8]:
client.search_registered_models()

[]

In [37]:
client.search_runs(experiment_ids='1')[0].data.tags['mlflow.log-model.history']

'[{"run_id": "bbfb1cdb47ff41b6942e24c27958df24", "artifact_path": "models", "utc_time_created": "2024-07-05 12:07:37.246616", "flavors": {"python_function": {"model_path": "model.pkl", "predict_fn": "predict", "loader_module": "mlflow.sklearn", "python_version": "3.12.4", "env": {"conda": "conda.yaml", "virtualenv": "python_env.yaml"}}, "sklearn": {"pickled_model": "model.pkl", "sklearn_version": "1.5.1", "serialization_format": "cloudpickle", "code": null}}, "model_uuid": "bdb203236705428a9d1cd42cc7d97322", "mlflow_version": "2.14.1", "model_size_bytes": 789}]'

In [38]:
run_id = "bbfb1cdb47ff41b6942e24c27958df24"
mlflow.register_model(
    model_uri=f"runs:/{run_id}/models",
    name='iris-classifier'
)

Successfully registered model 'iris-classifier'.
2024/07/05 14:14:51 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: iris-classifier, version 1
Created version '1' of model 'iris-classifier'.


<ModelVersion: aliases=[], creation_timestamp=1720181691019, current_stage='None', description='', last_updated_timestamp=1720181691019, name='iris-classifier', run_id='bbfb1cdb47ff41b6942e24c27958df24', run_link='', source='mlflow-artifacts:/1/bbfb1cdb47ff41b6942e24c27958df24/artifacts/models', status='READY', status_message='', tags={}, user_id='', version='1'>